# 03 — Value Scoring & Deal Finder

This is the payoff notebook. We apply the trained hedonic model to **active listings**
in Saratoga Springs and Eagle Mountain, score each home's value, apply your criteria,
and surface the best deals.

**Run after:** `02_model.ipynb` (requires a saved model in `../models/`)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.scraper import load_all_raw_csv
from src.features import prepare_dataset
from src.model import load_model, predict
from src.scorer import compute_value_scores, top_deals, score_summary
from src.filter import apply_criteria, filter_summary, load_config
from src.notifier import print_deals_to_console, build_html_email

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
pd.set_option('display.float_format', '${:,.0f}'.format)

## 1. Load Data & Model

In [ ]:
config = load_config('../config.yaml')
target_cities = config['search']['cities']
print('Target cities:', target_cities)

raw = load_all_raw_csv('../data/raw')
df = prepare_dataset(raw)

# Active listings in target cities only
if 'sold' in df.columns:
    active = df[(df['sold'] == False) & (df['city'].isin(target_cities))].copy()
else:
    active = df[df['city'].isin(target_cities)].copy()

print(f'Active listings to score: {len(active):,}')
active[['city', 'price', 'beds', 'baths', 'sqft', 'year_built']].head()

In [ ]:
model, feature_cols = load_model(
    '../models/hedonic_model.joblib',
    '../models/hedonic_model_features.joblib',
)
print('Model loaded successfully.')
print(f'Model uses {len(feature_cols)} features.')

## 2. Predict & Score

In [ ]:
predicted_prices = predict(model, feature_cols, active)
scored = compute_value_scores(active, predicted_prices)

print(score_summary(scored))
scored[['address', 'city', 'price', 'predicted_price', 'pct_below_market', 'composite_score']].head(10)

## 3. Value Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Value score distribution
ax = axes[0]
ax.hist(scored['pct_below_market'].clip(-30, 30), bins=40, color='steelblue', edgecolor='white')
ax.axvline(0, color='black', linewidth=1.5, label='Fairly priced')
ax.axvline(5, color='green', linestyle='--', linewidth=1.5, label='5% below (alert threshold)')
ax.axvline(10, color='darkgreen', linestyle='--', linewidth=1.5, label='10% below (great deal)')
ax.set_xlabel('% Below Predicted Market Price')
ax.set_ylabel('Count')
ax.set_title('Value Score Distribution\n(Positive = underpriced, Negative = overpriced)')
ax.legend(fontsize=8)

# By city
ax = axes[1]
target_cities_present = [c for c in target_cities if c in scored['city'].values]
colors = ['#3498db', '#e74c3c']
for city, color in zip(target_cities_present, colors):
    subset = scored[scored['city'] == city]['pct_below_market'].clip(-30, 30)
    ax.hist(subset, bins=20, alpha=0.6, color=color, edgecolor='white', label=city)
ax.axvline(0, color='black', linewidth=1.2)
ax.set_xlabel('% Below Predicted Price')
ax.set_title('Value Scores by City')
ax.legend()

plt.tight_layout()
plt.show()

## 4. Price vs Predicted — Scatter

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

# Color by value score
scatter = ax.scatter(
    scored['predicted_price'] / 1e3,
    scored['price'] / 1e3,
    c=scored['pct_below_market'].clip(-20, 20),
    cmap='RdYlGn',
    s=60, alpha=0.7, edgecolors='white', linewidths=0.4
)
plt.colorbar(scatter, ax=ax, label='% Below Market (Green = Deal, Red = Overpriced)')

# Perfect pricing line
lims_max = max(scored['predicted_price'].max(), scored['price'].max()) / 1e3 * 1.05
lims_min = min(scored['predicted_price'].min(), scored['price'].min()) / 1e3 * 0.95
ax.plot([lims_min, lims_max], [lims_min, lims_max], 'k--', linewidth=1.2, label='Fairly priced')

ax.set_xlabel('Predicted Price ($K)')
ax.set_ylabel('Actual Listing Price ($K)')
ax.set_title('Predicted vs Listed Price\n(Green dots below the line = underpriced deals)')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Apply Your Criteria Filter

In [ ]:
print('Applying criteria from config.yaml...')
filtered = apply_criteria(scored, config)
print(filter_summary(scored, filtered, config))

## 6. Top Deals — The Homes Worth Investigating

In [ ]:
scoring_cfg = config.get('scoring', {})
min_value_score = scoring_cfg.get('min_value_score', 0.05)
top_n = scoring_cfg.get('top_n_alerts', 5)

deals = top_deals(filtered, min_value_score=min_value_score, top_n=top_n)
print_deals_to_console(deals)

## 7. Deals Table

In [ ]:
display_cols = ['address', 'city', 'price', 'predicted_price', 'pct_below_market',
                'beds', 'baths', 'sqft', 'year_built', 'days_on_market',
                'composite_score', 'url']
display_cols = [c for c in display_cols if c in deals.columns]

deals[display_cols].style.format({
    'price': '${:,.0f}',
    'predicted_price': '${:,.0f}',
    'pct_below_market': '{:.1f}%',
    'sqft': '{:,.0f}',
    'composite_score': '{:.3f}',
}).background_gradient(subset=['composite_score'], cmap='Greens')

## 8. Composite Score vs Price — Full Ranked View

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for city, color in zip(target_cities_present, colors):
    subset = filtered[filtered['city'] == city]
    ax.scatter(
        subset['price'] / 1e3,
        subset['composite_score'],
        color=color, alpha=0.6, s=50, edgecolors='white', linewidths=0.4, label=city
    )

# Highlight top deals
if len(deals) > 0:
    ax.scatter(
        deals['price'] / 1e3,
        deals['composite_score'],
        color='gold', s=150, zorder=5, edgecolors='black', linewidths=1.5,
        label=f'Top {len(deals)} deals',
        marker='*'
    )

ax.set_xlabel('Listing Price ($K)')
ax.set_ylabel('Composite Value Score')
ax.set_title('Composite Score vs Price\n(Gold stars = your top deals)')
ax.legend()
plt.tight_layout()
plt.show()

## 9. Preview Email Alert

In [ ]:
from IPython.display import HTML

html = build_html_email(deals)
HTML(html)

## 10. Tune Your Criteria

Edit `../config.yaml` and re-run cells 5–8 above to see how different criteria affect your deal list.

Key levers:
- `max_price` — raise or lower to widen/narrow the pool
- `min_sqft` — how much space do you really need?
- `max_home_age_years` — newer homes = less maintenance but higher price
- `min_value_score` — lower to 0.03 if you're not seeing enough deals